# LangChain Basics — Prompts, LLMs & Chains

The three foundational building blocks of every LangChain application.

---

In [ ]:
!pip install -q langchain langchain-openai langchain-anthropic

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 1 · LLM Wrappers — Unified Interface Across Providers

LangChain abstracts away provider differences. Same `.invoke()`, different model.

In [ ]:
# Initialize — swap models without changing downstream code
gpt = ChatOpenAI(model="gpt-4o-mini", temperature=0)
claude = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)

# Identical interface
for name, llm in [("GPT", gpt), ("Claude", claude)]:
    resp = llm.invoke("What is LangChain in one sentence?")
    print(f"{name}: {resp.content}\n")

## 2 · Prompt Templates — Reusable, Parameterized Prompts

Separate prompt logic from application logic. Define once, reuse everywhere.

In [ ]:
# Basic template with variables
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Be concise — 2 sentences max."),
    ("human", "{question}")
])

# Inspect the formatted output
messages = prompt.format_messages(
    role="senior ML engineer",
    question="When should I use RAG vs fine-tuning?"
)
for m in messages:
    print(f"[{m.type}] {m.content}")

### Few-Shot Prompting

Provide examples to steer the model's output format and reasoning style.

In [ ]:
# Define examples
examples = [
    {"input": "What is gradient descent?",
     "output": "CONCEPT: Gradient Descent\nCATEGORY: Optimization\nTLDR: Iteratively adjust parameters by moving in the direction of steepest loss reduction."},
    {"input": "What is dropout?",
     "output": "CONCEPT: Dropout\nCATEGORY: Regularization\nTLDR: Randomly deactivate neurons during training to prevent co-adaptation and overfitting."},
]

# Build the few-shot prompt
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an ML glossary bot. Follow the exact format shown."),
    few_shot,
    ("human", "{input}")
])

# Test it
chain = final_prompt | gpt | StrOutputParser()
print(chain.invoke({"input": "What is batch normalization?"}))

## 3 · Chains — The Pipe Operator

The `|` operator connects components into a pipeline: `prompt → model → parser`.

This is **LCEL** (LangChain Expression Language) — covered in depth in Tutorial 02.

In [ ]:
# Minimal chain
chain = prompt | gpt | StrOutputParser()

result = chain.invoke({
    "role": "data scientist",
    "question": "When should I use XGBoost vs a neural network?"
})
print(result)

In [ ]:
# Swap the model — zero code changes elsewhere
chain_claude = prompt | claude | StrOutputParser()

result = chain_claude.invoke({
    "role": "data scientist",
    "question": "When should I use XGBoost vs a neural network?"
})
print(result)

## 4 · Sequential Chains — Multi-Step Pipelines

Chain multiple LLM calls together. Output of step 1 feeds into step 2.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Step 1: Generate a technical explanation
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer."),
    ("human", "Explain {concept} in 3 sentences for an engineer.")
])

# Step 2: Simplify for a non-technical audience
simplify_prompt = ChatPromptTemplate.from_messages([
    ("system", "You simplify technical concepts for business executives."),
    ("human", "Rewrite this for a non-technical CEO in 2 sentences:\n\n{technical_explanation}")
])

# Chain them: concept → technical explanation → simplified version
step1 = explain_prompt | gpt | StrOutputParser()
step2 = simplify_prompt | gpt | StrOutputParser()

full_chain = (
    {"technical_explanation": step1}  # step1 output → step2 input
    | step2
)

result = full_chain.invoke({"concept": "vector embeddings"})
print(result)

## 5 · Batch & Streaming

Process multiple inputs at once, or stream tokens in real-time.

In [ ]:
# Batch — process multiple inputs in parallel
chain = prompt | gpt | StrOutputParser()

results = chain.batch([
    {"role": "ML engineer", "question": "What is attention?"},
    {"role": "ML engineer", "question": "What is a transformer?"},
    {"role": "ML engineer", "question": "What is RLHF?"},
])

for i, r in enumerate(results, 1):
    print(f"Q{i}: {r}\n")

In [ ]:
# Streaming — token-by-token output
print("Streaming: ", end="")
for chunk in chain.stream({"role": "poet", "question": "Describe transformers"}):
    print(chunk, end="", flush=True)
print()  # newline

---

## Key Takeaways

| Concept | Code | What It Does |
|---------|------|--------------|
| LLM Wrapper | `ChatOpenAI()`, `ChatAnthropic()` | Unified `.invoke()` across providers |
| Prompt Template | `ChatPromptTemplate.from_messages()` | Reusable prompts with variables |
| Few-Shot | `FewShotChatMessagePromptTemplate` | Steer output format with examples |
| Chain (LCEL) | `prompt \| llm \| parser` | Pipe operator connects components |
| Sequential Chain | `{"key": chain1} \| chain2` | Multi-step LLM pipelines |
| Batch | `chain.batch([...])` | Parallel processing |
| Stream | `chain.stream({...})` | Real-time token output |

**Next →** [02 · LCEL Deep Dive](../02-lcel-deep-dive/)

---
*Part of the [LangChain Tutorials](https://github.com/hitpant/langchain-tutorials) series by [Hitesh Pant](https://github.com/hitpant)*